# 10 -- LLM + MCTS (plan v5 Step 17): timing probe, control, then self-play training

Step 17 asks: from the same SFT start (the 500k run's step 30k, end of pass 1), does **self-play with search**
go further than **more imitation** (SFT pass 2) for the same extra GPU-hours?

This notebook runs it in three parts:
1. **Timing probe** (~5 min): how fast the LLM drives a batched search on this GPU -- one search move across 64
   games at 8 and 16 simulations. Turns the plan's throughput estimate into a measurement.
2. **Control** (~2 h at 16 simulations, ~1 h at 8): the start checkpoint rated **with search at play time**, no
   extra training, against the same opponents as its no-search rating. Its Elo gain over no-search is the go /
   no-go check for part 3: a clear gain means the search's move choices beat the policy's, so they are worth
   training on.
3. **Self-play training (arm 1)**: built after the control (plan Step 17, gate before launch).

Switches come from the shell: `PROBE=1`, `CONTROL_SIMS=16` (or `16,8`; empty = skip). Every control game goes to
`results/games.csv`; a rerun skips a control rating that already has its full games. Needs a GPU pod.

In [ ]:
# Setup (RunPod): the repo lives on the volume; artifacts under artifact_root (configs/base.yaml).
import os
_here = os.getcwd()
REPO_DIR = os.path.dirname(_here) if os.path.basename(_here) == "notebooks" else _here
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements.txt

import json
import yaml
import torch
ROOT = yaml.safe_load(open("configs/base.yaml"))["artifact_root"]
from engine.stockfish import ensure_stockfish, SF_VERSION
print("Stockfish:", SF_VERSION, ensure_stockfish())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none -- this needs a GPU pod"
print("GPU:", GPU)

## Settings
The start checkpoint (Step 17's phase 1) and the control's opponents: the same as the 500k checkpoints'
no-search ratings (runbook I), so the two rows compare directly.

In [ ]:
PROBE = os.environ.get("PROBE", "1") == "1"
CONTROL_SIMS = [int(x) for x in os.environ.get("CONTROL_SIMS", "16").split(",") if x.strip()]
START_RUN, START_STEP = "structured_masked_nall_e2_s0_scaled_d12_n500000", 30000
START = {"name": f"{START_RUN}@{START_STEP}", "kind": "sft", "run": START_RUN, "step": START_STEP}
OPPONENTS = ["greedy", "step_20000", "sf_1320"]
DATA_DIR = os.path.join(ROOT, "data", "processed", "sft_scaled_d12_n500000")
print("start:", START["name"], "| probe:", PROBE, "| control simulations:", CONTROL_SIMS, "| opponents:", OPPONENTS)

## 1. Timing probe
Batched evaluation speed (positions per second at 32 / 64 / 128 positions per call), then one search move across
64 games at 8 and 16 simulations. The projection uses the plan's estimate for the training update (~0.2 s per
position); part 3's smoke test measures that too.

In [ ]:
import random
import chess
import time
from eval.agents import checkpoint_dir
from eval.match import random_opening
from model.llm_evaluator import LLMEvaluator, time_search
from train.sft import load_checkpoint

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

if PROBE:
    policy = load_checkpoint(checkpoint_dir(START, ROOT), name=START["name"])
    ev = LLMEvaluator(policy, batch_positions=128)
    rng = random.Random(0)
    boards = []
    for _ in range(128):
        b = chess.Board()
        for m in random_opening(rng, 12):
            b.push_uci(m)
        boards.append(b)
    ev.evaluate(boards[:8])                                    # warm-up (CUDA kernels, caches)
    for n in (32, 64, 128):
        sync(); t0 = time.perf_counter()
        ev.evaluate(boards[:n])
        sync(); dt = time.perf_counter() - t0
        print(f"evaluate {n:>3} positions: {dt:.2f}s -> {1000 * dt / n:.1f} ms/position")
    probe = {}
    for sims in (8, 16):
        r = time_search(ev, boards[:64], sims)
        probe[sims] = r
        per_position = r["seconds_per_game_move"] + 0.2           # + the update's estimate
        n_pos = 9.6 * 3600 / per_position
        print(f"{sims} simulations: one move of 64 games {r['seconds']:.1f}s -> {r['seconds_per_game_move']:.3f}s per "
              f"game-move ({1000 * r['seconds_per_position']:.1f} ms/position, {r['model_calls']} model calls); "
              f"~{n_pos / 1000:.0f}k training positions (~{n_pos / 100:.0f} games) in 9.6 GPU-h")
    if torch.cuda.is_available():
        print("peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")
    del policy, ev
    torch.cuda.empty_cache()
else:
    print("PROBE is off")

## 2. Control: the start checkpoint with search at play time
Rows `<run>_mcts<N>@<step>` (their own series), carrying the start checkpoint's training GPU-hours: search adds
no training compute, its cost shows in sec/move. ~1.5 s/move at 16 simulations.

In [ ]:
from eval.agents import agent_name, is_rerated, rate
from eval.anchors import ResultsTable, build_reference_agents, close_agents
from eval.elo import format_table
from train.sft import gpu_hours_at

eval_cfg = yaml.safe_load(open("configs/eval.yaml"))
GAMES, BATCH = eval_cfg["games_per_opponent"], eval_cfg["batch_games"]
results = ResultsTable(os.path.join(ROOT, eval_cfg["results_file"]))
todo = [s for s in CONTROL_SIMS if not is_rerated(results, agent_name(START["name"], s), GAMES * len(OPPONENTS))]
for s in CONTROL_SIMS:
    if s not in todo:
        print(f"{agent_name(START['name'], s)}: already rated with its full games -- skipped")
if todo:
    run_dir = os.path.join(ROOT, "checkpoints", "sft", START_RUN)
    label_cpu = json.load(open(os.path.join(DATA_DIR, "manifest.json")))["stockfish_cpu_seconds"] / 3600
    opponents = build_reference_agents(results, OPPONENTS, ROOT)
    try:
        for sims in todo:
            name = agent_name(START["name"], sims)
            print(f"\n== {name} vs {OPPONENTS}")
            est = rate({**START, "name": name, "simulations": sims}, results, ROOT, opponents,
                       fields={"run": f"{START_RUN}_mcts{sims}", "step": START_STEP, "seed": 0,
                               "train_gpu_hours": round(gpu_hours_at(run_dir, START_STEP), 4),
                               "label_cpu_hours": round(label_cpu, 4), "dataset_size": 500000,
                               "note": f"Step 17 control: 500k SFT step {START_STEP} + MCTS {sims} sims at play, no extra training"},
                       games_per_opponent=GAMES, batch_games=BATCH, gpu=GPU)
            print(format_table([est]))
            torch.cuda.empty_cache()
    finally:
        close_agents(opponents)

## Search gain
The start checkpoint without search (from runbook I's evals, if done) next to each control row. The gain is the
go / no-go number for part 3.

In [ ]:
import pandas as pd

t = ResultsTable(results.path)
names = [START["name"]] + [agent_name(START["name"], s) for s in CONTROL_SIMS]
rows = pd.DataFrame([t.rows[n] for n in names if n in t.rows])
display(rows[["name", "rating", "ci_low", "ci_high", "games", "wins", "draws", "losses", "sec_per_move"]])
if START["name"] in t.rows:
    base = float(t.rows[START["name"]]["rating"])
    for s in CONTROL_SIMS:
        n = agent_name(START["name"], s)
        if n in t.rows:
            print(f"{s} simulations add {float(t.rows[n]['rating']) - base:+.0f} Elo over no search")
else:
    print("no no-search row for the start checkpoint yet -- rate it first (runbook I)")

## 3. Self-play training (arm 1)
Built after the control (plan Step 17: if 16 simulations add a clear Elo gain, the trainer `train/llm_mcts.py`
is built, smoke-tested here, then run to the same GPU-hours as SFT pass 2 on its own pod).